# End to end 5 — twelve studies in, one prior out

Twelve studies report the same dimensionless quantity — the standardized contrast at
saturation of a fertilizer dose — from experiments and from fitted models, at several
labs. This notebook takes them through `axiom.meta` to one pooled prior and puts that
prior back into a response surface.

| step | subpackage |
|---|---|
| build the corpus; refuse a study on the wrong scale, then admit it under a licensed plan | `build.MetaBuilder`, `meta.contribute`, `meta.ingest` |
| classical pooling: three τ estimators, heterogeneity, prediction interval, Egger, leave-one-out | `meta.classical`, `meta.influence` |
| Bayesian pooling with a model-vs-experiment bias term δ | `meta.pool` |
| the pooled predictive as a lognormal amplitude prior | `meta.priors` |
| a surface refit with and without the pooled prior | `surface`, `sim` |
| a privacy-gated release of the pooled cell | `meta.privacy`, `meta.publish` |

In [ ]:
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd

from axiom.build import MetaBuilder
from axiom.core import Posterior, Unsupported
from axiom.estimands import TransferPlan
from axiom.io import Analysis, load_analysis, save_analysis
from axiom.meta import (
    Corpus, EpsilonLedger, PoolPriors, PoolSpec, Pooled, PrivacyPolicy, Release, cell_from_records, check_cell,
    delta_identification, egger, heterogeneity, leave_one_out, normalize, pool, prediction_interval,
    prior_from_pool, random_effects, record_from_summary, release,
)
from axiom.sim import DosePlan, surface_world
from axiom.surface import HillKernel, fit

from IPython.display import display
from axiom.display import enable, table

enable();  # every axiom result renders itself from here on

pd.set_option("display.width", 160)

## The studies

Eleven come in through `MetaBuilder`: id, estimate, standard error, whether the number
is an `experiment` read or a `model` read, the lab that produced it, its size, and a
moderator (follow-up length). Two labs (`lab_a`, `lab_b`) contributed *both* reads —
these dual-read contributors are what identifies the model-vs-experiment bias later.

The twelfth study reported its contrast *per acre*. `normalize` refuses it: pooling is
over dimensionless quantities, and a number on a scale needs a licensed `TransferPlan`
before it can enter.

In [ ]:
rng = np.random.default_rng(5)
rows = [
    ("s01", "lab_a", "experiment", 0.62, 0.08, 120), ("s02", "lab_a", "model", 0.81, 0.06, 400),
    ("s03", "lab_b", "experiment", 0.55, 0.12, 60), ("s04", "lab_b", "model", 0.74, 0.07, 300),
    ("s05", "lab_c", "experiment", 0.70, 0.10, 90), ("s06", "lab_d", "experiment", 0.48, 0.15, 40),
    ("s07", "lab_e", "model", 0.77, 0.05, 500), ("s08", "lab_f", "experiment", 0.66, 0.09, 100),
    ("s09", "lab_g", "model", 0.85, 0.08, 250), ("s10", "lab_h", "experiment", 0.58, 0.11, 70),
    ("s11", "lab_i", "experiment", 0.64, 0.07, 150),
]
mb = MetaBuilder().name("saturation-contrast").family("fertilizer")
for study, lab, read, est, se, n in rows:
    mb = mb.study(study, estimate=est, se=se, read=read, contributor=lab, quantity="standardized_contrast",
                  n=n, moderators={"follow_up": float(rng.integers(4, 13))}, source=f"report-{study}")
eleven = mb.build()
print(eleven.name, "| k =", len(eleven), "| dual-read contributors:", eleven.dual_read_contributors("fertilizer"))

per_acre = record_from_summary(
    study="s12", contributor="lab_j", quantity="standardized_contrast", estimate=1.32, se=0.20,
    read="experiment", family="fertilizer", n=80, unit_scale="per_acre", moderators={"follow_up": 6.0},
    source="report-s12",
)
refused = normalize([*eleven.records, per_acre], name="twelve")
assert isinstance(refused, Unsupported)
print("\nrefused:", refused.reason)
print("missing:", refused.missing)

### Admitting the twelfth under a licensed plan

A `TransferPlan` whose status licenses the read admits the record; `normalize` stamps the
plan's hash and status into the record's `detail` so the corpus carries the license.

> **Gap, worked around here.** `normalize` admits the record but leaves its
> `unit_scale` set, and `pool` refuses any record whose `unit_scale` is non-empty — so
> the admitted corpus is still unpoolable. The conversion the plan licenses (here,
> the study's per-acre contrast divided by the reference area of 2 acres) is applied
> by hand below, with the admission detail kept on the record. Either `normalize`
> should apply and clear the scale on admission, or `pool` should honour the
> admission detail; that decision belongs to `axiom.meta`.

In [ ]:
plan = TransferPlan(status="identified", source="per_acre", target="dimensionless",
                    differing=(), entries=(), assumptions=(), ledger_lines=())
admitted = normalize([*eleven.records, per_acre], plans={"s12": plan}, name="twelve")
assert isinstance(admitted, Corpus)
s12 = admitted.records[-1]
print("admitted:", s12.study, "|", s12.detail["transfer"], "| plan", s12.detail["transfer_plan_hash"][:12])

REFERENCE_AREA = 2.0  # acres; the scale the plan licenses dividing out
converted = s12.model_copy(update={
    "estimate": s12.estimate / REFERENCE_AREA, "se": s12.se / REFERENCE_AREA, "unit_scale": "",
    "detail": {**s12.detail, "converted_by": f"/ {REFERENCE_AREA} acre"},
})
corpus = Corpus(records=(*admitted.records[:-1], converted), name="twelve")
print("poolable:", all(r.is_dimensionless for r in corpus.records), "| k =", len(corpus))
corpus.to_frame()[["study", "contributor", "read", "estimate", "se", "n"]]

## Classical pooling

`random_effects` with each of the three τ² estimators (DerSimonian–Laird, Paule–Mandel,
REML), the heterogeneity statistics, the prediction interval for a new study, Egger's
small-study test, and the leave-one-out influence of each study on the pooled mean.

In [ ]:
y, se = corpus.arrays()
rows = []
for method in ("dl", "pm", "reml"):
    re = random_effects(y, se, tau_method=method)
    rows.append([method, f"{re.estimate:.4f}", f"{re.se:.4f}", f"{re.tau2:.5f}", str(re.interval)])
table(rows, headers=("tau method", "mu", "se", "tau²", "interval"))
het = heterogeneity(y, se)
print(f"\nQ = {het.q:.2f} on {het.df} df (p = {het.p_value:.3f}), I² = {het.i2:.2f}")
kh = random_effects(y, se, tau_method="reml", knapp_hartung=True)
print("prediction interval for a new study:", prediction_interval(kh))
eg = egger(y, se)
print(f"Egger intercept {eg.intercept:+.3f} ± {eg.se:.3f}, t = {eg.t:+.2f} on {eg.df} df, p = {eg.p:.3f}")
loo = leave_one_out(y, se, method="reml")
influence = pd.Series(loo.influence, index=[r.study for r in corpus.records]).round(3)
print("\nleave-one-out influence (z of the shift):")
display(influence.sort_values().to_frame("influence"))

## Bayesian pooling with a bias term

Model reads and experiment reads of the same quantity need not agree: a fitted model can
carry a systematic offset. `PoolSpec(bias_term=True)` adds δ, the model-minus-experiment
offset, which is identified only because two contributors reported both reads.
`delta_identification` checks that before `pool` runs.

In [ ]:
verdict = delta_identification(corpus, "fertilizer")
print("delta identification:", verdict.status, "|", verdict.route)
pool_spec = PoolSpec(family="fertilizer", bias_term=True,
                     priors=PoolPriors(mu_scale=2.0, tau_scale=0.5, delta_scale=1.0), mass=0.9, definition="eti")
pooled = pool(pool_spec, corpus, backend="laplace", draws=2000, seed=0)
assert isinstance(pooled, Pooled) and isinstance(pooled.posterior, Posterior)
res = pooled.result
print(f"mu    {res.mu.mean:.4f} ± {res.mu.sd:.4f}  {res.mu.interval}   (the experiment-scale mean)")
print(f"tau   {res.tau.mean:.4f} ± {res.tau.sd:.4f}")
print(f"delta {res.delta.mean:+.4f} ± {res.delta.sd:.4f}  identified={res.delta.identified}   (model reads sit this much above experiments)")
print("k =", res.k, "| backend:", res.backend)
print("classical REML mu without the bias term:", round(random_effects(y, se, tau_method="reml").estimate, 4))

## The pooled prior, back into a surface

`prior_from_pool` turns the pooled *predictive* — what a new study would see, τ
included — into a `Prior`. As a lognormal it is a legal `HillKernel` amplitude prior.
A small new panel (three units, ten periods, outcome in standard-deviation units so
the amplitude *is* the standardized contrast) is fitted twice: with the kernel's default
weakly-informative amplitude prior and with the pooled one.

In [ ]:
amplitude_prior, handoff_line = prior_from_pool(res, target="predictive", family="lognormal")
print(amplitude_prior)
print(handoff_line.kind, "|", handoff_line.statement)

TRUTH_BETA = 0.6
world = surface_world(
    n_units=3, n_periods=10, treatments=("a",), kernels=HillKernel(reference_dose=50.0, amplitude_scale=1.0),
    doses=DosePlan(scale=50.0, spread=0.8), intercept="shared",
    truth={"beta_a": TRUTH_BETA, "k_a": 50.0, "s_a": 2.0, "alpha": 0.0}, noise_sd=1.0, seed=2,
)
informed = world.spec.model_copy(update={
    "kernels": {"a": HillKernel(reference_dose=50.0, amplitude_scale=1.0, amplitude_prior=amplitude_prior)}
})
print("default kernel:", world.spec.kernel_of("a"))
print("informed kernel:", informed.kernel_of("a"))

In [ ]:
without = fit(world.spec, world.panel, backend="laplace", draws=1000, seed=0)
with_pool = fit(informed, world.panel, backend="laplace", draws=1000, seed=0)
rows = []
for label, result in (("default prior", without), ("pooled prior", with_pool)):
    assert isinstance(result.posterior, Posterior) and result.converged
    b = result.posterior.flat("beta_a")
    rows.append([label, f"{b.mean():.3f}", f"{b.std(ddof=1):.3f}", TRUTH_BETA])
table(rows, headers=("fit", "beta_a", "sd", "truth"))
print("the pooled prior's own mean:", round(float(np.exp(amplitude_prior.hyper['mu'] + amplitude_prior.hyper['sigma'] ** 2 / 2)), 3))

## Releasing the pooled cell

Before the cell's mean leaves the consortium it passes the privacy policy — at least `k`
contributors, no single contributor dominating, no top-two dominating — and a
differentially private mechanism charges the ε budget. The `Release` records the
mechanism, the clip, the noise scale, and the seed, so the published number is
reproducible and its budget is accounted for.

In [ ]:
cell = cell_from_records(corpus.records, name="fertilizer/standardized_contrast")
policy = PrivacyPolicy(k=3, dominance_p=0.5, dominance_top_n=2, dominance_top_p=0.8, epsilon_total=1.0)
gate = check_cell(cell, policy)
print("policy check:", gate.status, "|", gate.route, "| contributors:", len(cell.contributors))
out = release(cell, policy, EpsilonLedger(budget=policy.epsilon_total), release_id="fertilizer-2026Q3",
              epsilon=0.5, clip=(0.0, 1.5), seed=0, mechanism="laplace")
assert isinstance(out, tuple)
published, budget = out
assert isinstance(published, Release)
print(f"released {published.value:.4f}  {published.interval}  (clipped mean {np.mean(np.clip(cell.values, 0, 1.5)):.4f})")
print(f"  mechanism {published.mechanism}, scale {published.noise_scale:.4f}, ε = {published.epsilon}; budget used {budget.used}, remaining {budget.remaining}")

## The meta contribution, saved

The corpus, the pool spec and result, the classical estimate, the release, and the
handoff line all serialize; the pooled posterior goes to `npz`. This is the
"meta contribution" part of an analysis directory.

In [ ]:
analysis = (
    Analysis(specs={"corpus": corpus, "pool": pool_spec, "pool:result": res, "classical:reml": kh,
                    "release": published, "surface:informed": informed})
    .with_posterior(pooled.posterior)
    .with_ledger_line(handoff_line)
)
root = Path(tempfile.mkdtemp()) / "twelve-studies.axiom"
saved = save_analysis(analysis, root, seed=0)
loaded = load_analysis(root)
print("equal after round-trip:", loaded == saved, "| hashes identical:", loaded.hashes() == saved.hashes())
print("pooled mu identical:", loaded.spec("pool:result").mu.mean == res.mu.mean,
      "| released value identical:", loaded.spec("release").value == published.value)
print("no pickle on disk:", not any(p.suffix in (".pkl", ".pickle") for p in root.rglob("*")))